# Matrix Decompositions

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.04, 01.06, 01.07 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/08_matrix_decompositions.ipynb)

---

## 🎯 Learning Objective

Survey the major matrix factorizations — QR, Cholesky, LU, Schur — and learn which decomposition to reach for depending on the structure of your problem.

---

## 📐 Theory

Different matrix structures call for different factorizations. A quick field guide:

**QR decomposition**: $A = QR$, with $Q$ orthogonal and $R$ upper triangular. Built via
Gram-Schmidt orthogonalization (or the numerically sturdier Householder reflections). QR is
the standard, numerically stable way to solve least-squares problems without ever forming
$A^\top A$ (which, as in the normal equations of
[Notebook 01.04](04_systems_of_equations.ipynb), can lose precision).

**Cholesky decomposition**: for symmetric positive-definite $A$, $A = LL^\top$ with $L$ lower
triangular. It's roughly twice as fast as general-purpose factorizations *because* it exploits
positive-definiteness, and it's the standard way to sample from a multivariate Gaussian with
covariance $A$.

**LU decomposition**: $A = LU$ (or $PA = LU$ with a row-permutation $P$ for numerical
stability), the "packaged" output of Gaussian elimination introduced in
[Notebook 01.04](04_systems_of_equations.ipynb).

**Schur decomposition**: $A = QTQ^\top$ (or with complex entries, $A=UTU^*$) where $T$ is upper
triangular and $Q$/$U$ is orthogonal/unitary — this always exists, even for non-symmetric $A$
where eigendecomposition might not, making it the general-purpose workhorse many library
eigenvalue routines actually compute internally.

### Choosing a decomposition

| Situation | Reach for |
|---|---|
| Solving $A\mathbf{x}=\mathbf{b}$ once | LU |
| Solving with many different $\mathbf{b}$, or least-squares | QR |
| $A$ symmetric positive-definite (e.g. a covariance matrix) | Cholesky |
| Need eigenvalues, $A$ not symmetric | Schur |

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Visualizing the factor matrices themselves as heatmaps shows their structure directly: QR's
$R$ is upper triangular, Cholesky's $L$ is lower triangular, and both are "sparser looking"
than the original dense matrix.

In [ ]:
# Heatmaps of the factor matrices reveal their triangular structure directly
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla

rng = np.random.default_rng(0)
B = rng.standard_normal((5, 5))
A_spd = B @ B.T + 5 * np.eye(5)   # guaranteed symmetric positive-definite

Q, R = np.linalg.qr(B)
L_chol = np.linalg.cholesky(A_spd)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, M) in zip(axes, [("R (upper triangular, from QR)", R),
                                   ("L (lower triangular, from Cholesky)", L_chol),
                                   ("Original SPD matrix A", A_spd)]):
    im = ax.imshow(M, cmap="coolwarm")
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Gram-Schmidt orthogonalization builds QR column by column, and a simple algorithm builds
Cholesky row by row — both checked against SciPy/NumPy.

In [ ]:
# QR via Gram-Schmidt, and Cholesky via its defining recursive formula
import numpy as np

def qr_gram_schmidt(A):
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]           # subtract off the component already covered by earlier columns
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def cholesky_from_scratch(A):
    n = A.shape[0]
    L = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i, k] * L[j, k] for k in range(j))
            if i == j:
                L[i, j] = np.sqrt(A[i, i] - s)
            else:
                L[i, j] = (A[i, j] - s) / L[j, j]
    return L

rng = np.random.default_rng(1)
B = rng.standard_normal((5, 4))
Q_scratch, R_scratch = qr_gram_schmidt(B)
Q_np, R_np = np.linalg.qr(B)
print("Gram-Schmidt QR reconstructs B:  ", np.allclose(Q_scratch @ R_scratch, B))
print("Q is orthogonal (Q^T Q = I):      ", np.allclose(Q_scratch.T @ Q_scratch, np.eye(4)))

C = rng.standard_normal((4, 4))
A_spd = C @ C.T + 4 * np.eye(4)
L_scratch = cholesky_from_scratch(A_spd)
L_np = np.linalg.cholesky(A_spd)
print("\nCholesky from scratch matches np.linalg.cholesky:", np.allclose(L_scratch, L_np))
print("L @ L^T reconstructs A:                            ", np.allclose(L_scratch @ L_scratch.T, A_spd))

## 🤖 Why This Matters for AI

**Cholesky decomposition is THE standard way to sample from a multivariate Gaussian.** Given a
covariance matrix $\Sigma = LL^\top$, you can turn independent standard-normal samples $z$
into correctly-correlated samples via $\mathbf{x} = \mu + Lz$. This exact trick underlies the
**reparameterization trick** in Variational Autoencoders, which is what makes it possible to
backpropagate through a random sampling step at all.

In [ ]:
# The reparameterization trick: sampling a correlated Gaussian via Cholesky, differentiably
import numpy as np

rng = np.random.default_rng(0)
mu = np.array([2.0, -1.0])
Sigma = np.array([[2.0, 1.2], [1.2, 1.5]])    # target covariance -- off-diagonal means correlated dims
L = np.linalg.cholesky(Sigma)

n_samples = 5000
z = rng.standard_normal((n_samples, 2))        # cheap to sample: independent standard normals
x = mu + z @ L.T                                # x = mu + L @ z  (the reparameterization trick)

print("Target covariance:\n", np.round(Sigma, 3))
print("Empirical covariance of samples:\n", np.round(np.cov(x.T), 3))
print("\nTarget mean:", mu, " Empirical mean:", np.round(x.mean(axis=0), 3))
print("\nCrucially: x = mu + L @ z is a DIFFERENTIABLE function of mu and L (hence of Sigma).")
print("This is exactly why VAEs can backprop through a 'random' sampling step during training.")

## 🏋️ Exercises

### Warm-up
1. For $A=\begin{pmatrix}4&2\\2&5\end{pmatrix}$, compute its Cholesky factor $L$ by hand using the recursive formula, then verify with `np.linalg.cholesky`.

### Practice
2. Use `qr_gram_schmidt` to factor a random $6\times3$ matrix, and use the resulting $Q,R$ to solve a least-squares problem $A\mathbf{x}\approx\mathbf{b}$ via back-substitution on $R\mathbf{x}=Q^\top\mathbf{b}$. Confirm it matches `np.linalg.lstsq`.

### Challenge
3. Compute the Schur decomposition of a random NON-symmetric $4\times4$ matrix with `scipy.linalg.schur`, and confirm the diagonal entries of the resulting (quasi-)triangular $T$ match `np.linalg.eigvals(A)` (accounting for possible complex-conjugate eigenvalue pairs appearing as 2x2 blocks). Explain in a sentence why Schur exists here even though this matrix has no valid eigendecomposition.

---

## 📚 Further Reading
- Trefethen & Bau, *Numerical Linear Algebra*, Lectures 7-8 (QR) and 23 (Cholesky)
- Kingma & Welling, ["Auto-Encoding Variational Bayes"](https://arxiv.org/abs/1312.6114) (the reparameterization trick)

---

*Next notebook: [Tensor Operations](09_tensor_operations.ipynb)*